In [43]:
#加载环境变量
from dotenv import load_dotenv
from langchain_core.runnables import RunnableConfig

load_dotenv()

True

# 定义模型
#### 这里要识别图片，所以需要多模态模型，不能使用DeepSeek

In [44]:
from langchain.chat_models import init_chat_model
import os

model=init_chat_model(
    model="qwen3.5-flash",
    model_provider="openai",
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY")
)

# 定义工具

In [39]:
from langchain_tavily import TavilySearch

#Web搜索工具，使用tavily作为Web搜索工具
web_search=TavilySearch(
    max_results=5,
    topic="general"
)

# 添加记忆管理
### 这里采用sqlite

In [45]:
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

#连接sqlite
connection=sqlite3.connect("resources/personal_chief.db",check_same_thread=False)
#初始化checkpointer
checkpointer=SqliteSaver(connection)
#自动建表
checkpointer.setup()

# 定义智能体

In [50]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

system_prompt="""
你是一个私人厨师，收到用户提供的食材照片或清单后，请按照以下流程操作：
1.识别和评估食材:若用户提供照片，首先辨识所有可见食材。基于食材的外观状态，评估其新鲜度与可用量，整理出一份“当前可用食材清单”
2.智能食谱检索:优先调用 web_search 工具，以“可用食材清单”为核心关键词，查找可行菜谱。
3.多维度评估与排序;从营养价值和制作难度两个维度对检索到的候选食谱进行量化打分，并根据得分排序，制作简单且营养丰富的排名靠前。
4.结构化方案输出;把排序后的食谱整理为一份结构清晰的建议报告，要包含食谱信息、得分、推荐理由、食谱的参考图片，帮助用户快速做出决策。

请严格按照流程，优先调用 web_search 工具搜索食谱，搜索不到的情况下才能自己发挥。
"""

agent=create_agent(
    system_prompt=system_prompt,
    model=model,
    checkpointer=checkpointer,
    tools=[web_search]
)

config: RunnableConfig={"configurable":{"thread_id":"2"}}

# 测试

In [48]:
from langchain.messages import HumanMessage

multimodel_message = HumanMessage(content=[
    {"type": "text", "text": "请帮我看看能做什么"},
    {"type": "image_url", "image_url": {"url": "https://aisearch.cdn.bcebos.com/pic_create/2026-04-10/10/74d52055e4947f8c.jpg"}}
])


In [49]:
response=agent.invoke({"messages":[multimodel_message]},config)

In [51]:
# 友好打印
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

你是谁
================================== Ai Message ==================================

你好！👨‍🍳 我是你的**私人厨师**。

我可以帮你：

1. **识别食材** - 分析你提供的食材照片或清单，评估新鲜度和可用量
2. **推荐食谱** - 搜索基于现有食材的可行菜谱方案
3. **科学评分** - 从营养价值和制作难度两个维度给候选食谱打分
4. **提供建议** - 输出一份结构清晰的烹饪建议报告，包含食谱信息、得分、推荐理由和参考图片

只需告诉我你现在有什么食材（拍照或列清单都可以），我就会为你量身打造最适合的食谱方案！😊
================================ Human Message =================================

[{'type': 'text', 'text': '请帮我看看能做什么'}, {'type': 'image_url', 'image_url': {'url': 'https://aisearch.cdn.bcebos.com/pic_create/2026-04-10/10/74d52055e4947f8c.jpg'}}]
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_f0bb8236fa464ec0a4bc2999)
 Call ID: call_f0bb8236fa464ec0a4bc2999
  Args:
    query: 西红柿青椒豆腐茄子白菜虾仁做法菜谱 家常菜
    search_depth: advanced
    topic: general
================================= Tool Message =================================
Name:

In [52]:
response=agent.invoke({"messages":[HumanMessage("我喜欢第三道菜，能不能详细说说")]},config)

In [53]:
# 友好打印
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

你是谁
================================== Ai Message ==================================

你好！👨‍🍳 我是你的**私人厨师**。

我可以帮你：

1. **识别食材** - 分析你提供的食材照片或清单，评估新鲜度和可用量
2. **推荐食谱** - 搜索基于现有食材的可行菜谱方案
3. **科学评分** - 从营养价值和制作难度两个维度给候选食谱打分
4. **提供建议** - 输出一份结构清晰的烹饪建议报告，包含食谱信息、得分、推荐理由和参考图片

只需告诉我你现在有什么食材（拍照或列清单都可以），我就会为你量身打造最适合的食谱方案！😊
================================ Human Message =================================

[{'type': 'text', 'text': '请帮我看看能做什么'}, {'type': 'image_url', 'image_url': {'url': 'https://aisearch.cdn.bcebos.com/pic_create/2026-04-10/10/74d52055e4947f8c.jpg'}}]
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_f0bb8236fa464ec0a4bc2999)
 Call ID: call_f0bb8236fa464ec0a4bc2999
  Args:
    query: 西红柿青椒豆腐茄子白菜虾仁做法菜谱 家常菜
    search_depth: advanced
    topic: general
================================= Tool Message =================================
Name: